# 📈 Estrategia de crecimiento para Bellabeat: análisis de patrones de actividad

> **Descargo de responsabilidad.** Este es un caso de estudio académico e independiente,
> realizado con fines de portafolio. **No está afiliado, patrocinado ni avalado por
> Bellabeat, Fitbit ni Google.** «Bellabeat» y «Fitbit» son marcas registradas de sus
> respectivos titulares y se usan aquí únicamente con fines identificativos y educativos.
> Las recomendaciones de la sección 11 no han sido solicitadas por Bellabeat y no
> representan la posición de la empresa.
>
> **No es consejo médico.** Los umbrales de la OMS y del JACC citados son directrices
> poblacionales, no criterios de diagnóstico individual. Los datos provienen de
> dispositivos de consumo no certificados como dispositivos médicos y con error de
> medición conocido. Las etiquetas de segmento («Sedentario», «Riesgo») son categorías
> **analíticas y de marketing**, no valoraciones clínicas de ninguna persona.

## 1. Introducción

### 1.1. Contexto del caso de estudio

Este proyecto es un caso de estudio estratégico para [**Bellabeat**](https://bellabeat.com), una empresa de tecnología enfocada en la salud y el bienestar de las mujeres. Bellabeat busca expandir su base de clientes y optimizar su estrategia de marketing y desarrollo de productos (como Leaf, Time y Spring) mediante el análisis de los hábitos de las personas que usan sus dispositivos.

Dado que Bellabeat opera con datos privados, este análisis utiliza el *dataset* público de **FitBit Fitness Tracker Data** como un **sustituto (proxy)** para inferir patrones de comportamiento que Bellabeat pueda priorizar.

> ⚠️ **El proxy tiene un límite importante y conviene declararlo desde el principio:** el
> dataset de FitBit **no contiene sexo, edad ni ubicación**. Por eso este documento usa
> lenguaje neutro («personas usuarias», «la muestra») en lugar del femenino: atribuir
> género a esta muestra sería asumir precisamente lo que el análisis pretende sustentar.
> La sección 2.4 detalla todas las limitaciones.

### 1.2. Preguntas guía y objetivos estratégicos

1.  **Comportamiento de consistencia:** ¿podemos segmentar a las personas usuarias según la **consistencia** de sus hábitos de salud?
2.  **Identificación de oportunidades:** ¿existen patrones de uso (ej., alta actividad con pocos pasos) que Bellabeat pueda abordar con mensajes específicos de **salud integral**?
3.  **Fidelización (*engagement*):** ¿qué perfiles son los más activos y cuáles los más **inconsistentes**? ¿Qué mensajes podrían *reenganchar* a quienes pierden el hábito?
4.  **Datos incompletos:** ¿qué proporción de personas aporta datos incompletos o intermitentes, lo que indica bajo *engagement* con el dispositivo?

### 1.3. Recursos de investigación en salud

Para responder con criterio es necesario establecer rangos saludables según investigación publicada, en lugar de umbrales populares sin respaldo (el conocido objetivo de «10 000 pasos» procede de una campaña publicitaria japonesa de 1965, no de evidencia clínica).

#### Pasos diarios — *Relationship of Daily Step Counts to All-Cause Mortality and Cardiovascular Events*

<https://www.jacc.org/doi/10.1016/j.jacc.2023.07.029>

Meta-análisis que cuantifica las asociaciones dosis-respuesta del recuento de pasos medido objetivamente en población general. De aquí salen los cortes de 2 735 / 4 930 / 7 126 / 10 000 pasos diarios.

#### Directrices de la OMS sobre actividad física y comportamientos sedentarios

<https://iris.who.int/items/65310979-92e8-4c98-8092-5a16ca07fc2f>

Recomienda entre **150 y 300 minutos** de actividad moderada semanal para personas de 18 a 64 años, y establece que **1 minuto de actividad vigorosa equivale a 2 minutos de actividad moderada**. De aquí salen tanto los cortes diarios de intensidad como la fórmula del puntaje combinado.

---
## 2. Preparación y comprensión de los datos

### 2.1. Fuente de datos (*dataset* proxy)

Se aplicó un proceso de Extracción, Transformación y Carga (ETL) al conjunto de datos de FitBit:

* **Correcciones técnicas aplicadas al dataset original:**
    * **Estandarización de unidades de tiempo** (corrección de errores de conversión entre minutos, horas y días).
    * **Validación de METs** y recategorización de los niveles de intensidad. Los valores de MET del dataset original vienen **multiplicados por diez** (el MET 1 de reposo aparece como 10), lo que invalidaba la clasificación de intensidad si se tomaba al pie de la letra.

Para usar los datos directamente: [FitBit Fitness Tracker Data: Corregido](https://www.kaggle.com/datasets/mvr513/fitbit-fitness-tracker-data-corrected).

Para procesar y exportar desde los archivos originales: [Fitbit-Data-Cleaning-ETL](https://github.com/MauricioVrx/Fitbit-Data-Cleaning-ETL).

Los registros originales corresponden a ***[FitBit Fitness Tracker Data](https://www.kaggle.com/datasets/arashnic/fitbit) (CC0: dominio público, publicado por Möbius)***.

| Elemento | Origen | Licencia |
| :--- | :--- | :--- |
| Datos crudos | Kaggle / Möbius, 2016 | CC0 1.0 |
| Datos corregidos | Este autor | CC0 1.0 |
| Código ETL | `MauricioVrx/Fitbit-Data-Cleaning-ETL` | MIT |
| Este análisis | Este repositorio | MIT (código) · CC BY 4.0 (contenido) |

### 2.2. Formato de los archivos

Los datos se organizan en carpetas por rango de fechas (`data/export_<inicio>-<fin>/`) que contienen los conjuntos **Intensidad, Calorías, Sueño y Pasos** en formatos **minuto, hora y día**, más el archivo `dailyActivity` con la actividad combinada diaria.

### 2.3. Comprensión de los datos

#### 2.3.1. Calorías
Calorías quemadas de forma basal más las derivadas de actividad física. Entre formatos solo se realiza una sumatoria **minuto → hora → día**.

#### 2.3.2. Pasos
Pasos detectados por el dispositivo. Igual sumatoria entre formatos.

#### 2.3.3. METs
**MET** significa *Metabolic Equivalent of Task* y estima el gasto energético de una actividad; 1 MET es la energía en reposo. En el dataset (original y corregido) los valores están **multiplicados por diez**. Solo se dispone del formato CSV por minuto; no se usa directamente en este análisis porque ya fue procesado en la fase de limpieza para derivar la Intensidad.

#### 2.3.4. Intensidad
Nivel de actividad física según el rango de METs:

| Intensidad (valor) | Rango de METs | Tipo de actividad |
| :--- | :--- | :--- |
| 0 | METs < 15 | **Sedentaria** / reposo |
| 1 | 15 ≤ METs < 35 | **Ligera** |
| 2 | 35 ≤ METs < 60 | **Moderada** |
| 3 | METs ≥ 60 | **Intensa** |

#### 2.3.5. `minutes_instances`

Número de registros por minuto efectivamente presentes en un día para esa persona (máximo 1 440). Es una **columna derivada por el pipeline de ETL**, no presente en el dataset original de FitBit. Se usa como medida de cobertura del dispositivo: `minutes_instances < 1440` indica minutos sin registro, ya sea por dispositivo descargado, retirado o sin sincronizar.

Se documenta aquí explícitamente porque de ella dependen tres cálculos posteriores: la completitud diaria, la estimación de calorías basales y los minutos faltantes de la cohorte de re-enganche.

### 2.4. Limitaciones y validez del análisis

Esta sección va **antes** de los resultados a propósito: condiciona cómo deben leerse todos los números que siguen.

Este análisis es un **caso de estudio con datos proxy**. Sus conclusiones son **hipótesis a validar con datos propios de Bellabeat**, no conclusiones transferibles directamente.

| Limitación | Alcance | Impacto sobre las conclusiones |
| :--- | :--- | :--- |
| **Sin datos demográficos** | El dataset no contiene sexo, edad ni ubicación | No se puede afirmar que la muestra represente a la base de clientas de Bellabeat, mayoritariamente femenina. **Es la limitación principal.** |
| **Muestra pequeña** | n = 26 tras filtrar (de 35 originales) | Cada persona mueve ≈ 3,8 puntos porcentuales. Los segmentos con n < 5 son ilustrativos, no concluyentes |
| **Antigüedad** | Datos de marzo–mayo de **2016** | Casi una década de cambio en hardware, hábitos y mercado post-pandemia |
| **Ventana corta y estacional** | 62 días, solo primavera boreal | Sin control de estacionalidad ni de tendencia anual |
| **Muestra autoseleccionada** | Voluntarios de Mechanical Turk que ya usaban Fitbit | Sesgo hacia personas más motivadas por el *fitness* que la población general |
| **Cohorte filtrada** | Solo personas con ≥ 57 días registrados | La completitud reportada es alta **por construcción**; el 26 % de la muestra fue excluida por uso intermitente (ver 6.2) |
| **Medidas repetidas** | Las correlaciones diarias se calculan sobre días-usuario | Existe pseudorreplicación: el n efectivo está más cerca de 26 que de 1 586. Se reportan como descriptivas, no inferenciales (ver 6.7) |
| **Dispositivo no médico** | Fitbit no está certificado como dispositivo médico | Los umbrales OMS/JACC son poblacionales, no diagnósticos individuales |

#### Qué haría falta para convertir esto en una decisión de negocio

1. Datos propios de Bellabeat con demografía (sexo, edad, país).
2. Un periodo de ≥ 6 meses para controlar estacionalidad.
3. n ≥ 200 personas por segmento para intervalos de confianza útiles.
4. Datos de conversión y retención para conectar segmento → valor de cliente.

---
## 3. Configuración

Las funciones de análisis viven en el paquete `src/bellabeat/` para que puedan probarse
de forma automática (`pytest tests/`). El notebook conserva la narrativa, las tablas y
los gráficos; el paquete conserva los cálculos de los que dependen las conclusiones.

Todos los umbrales están declarados en `bellabeat/config.py` con su fuente. No hay
números mágicos dentro del análisis.

In [ ]:
import sys
from pathlib import Path

# El paquete del proyecto vive en src/
sys.path.insert(0, str(Path.cwd() / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from bellabeat import config as cfg
from bellabeat import features, loading, plots, segmentation, stats

# Sistema visual único para todas las figuras del documento
plots.apply_style()
plots.reset_figure_numbers()

print("Umbrales en uso (todos derivados de literatura, ver config.py)")
print(f"  Intensidad diaria (min. efectivos) : {cfg.INTENSITY_BIN_EDGES[:-1]} -> {cfg.INTENSITY_LABELS}")
print(f"  Pasos diarios                      : {cfg.STEPS_BIN_EDGES[:-1]} -> {cfg.STEPS_LABELS}")
print(f"  Objetivo semanal de intensidad     : {cfg.WEEKLY_INTENSITY_TARGET:.0f} min "
      f"(techo OMS de {cfg.WHO_WEEKLY_OPTIMAL_MINUTES} min)")
print(f"  Mínimo semanal de intensidad       : {cfg.WEEKLY_INTENSITY_MINIMUM:.0f} min "
      f"(piso OMS de {cfg.WHO_WEEKLY_MINIMUM_MINUTES} min)")
print(f"  Objetivo semanal de pasos          : {cfg.WEEKLY_STEPS_TARGET:,.0f}")
print(f"  Sueño saludable                    : > {cfg.HEALTHY_SLEEP_MINUTES} min (7 h)")

---
## 4. Carga de datos

### 4.1. Registros por minuto

Se cargan Intensidad, Calorías y Pasos por minuto y se unen por `(Id, ActivityMinute)`.
La unión valida que la relación sea uno a uno: si hubiera claves duplicadas, el código
falla en lugar de multiplicar filas en silencio.

In [ ]:
minute_frames = {
    name: loading.load_activity_files(
        time_type="minute", info_type=name, time_column="ActivityMinute"
    )
    for name in ("Intensities", "Calories", "Steps")
}

df_minutes = loading.merge_minute_frames(
    minute_frames["Intensities"], minute_frames["Calories"], minute_frames["Steps"]
)

print(f"\nTotal: {len(df_minutes):,} minutos · {df_minutes['Id'].nunique()} usuarios")
df_minutes.head()

### 4.2. Correlación base: intensidad, pasos y gasto calórico

Correlación de Pearson entre las tres métricas **crudas por minuto**. El objetivo aquí es
verificar la **coherencia interna del registro del dispositivo**, no medir la métrica
derivada que se construye más adelante (esa se evalúa en la sección 6.7, con un valor
distinto y menor).

In [ ]:
corr_minute = df_minutes[["Intensity", "Calories", "Steps"]].corr()
corr_minute.round(3)

Las tres métricas base correlacionan fuertemente entre sí, lo que valida la coherencia
del registro:

* **Intensidad ↔ calorías:** la correlación más alta. El tiempo en actividad moderada o
  vigorosa es el principal motor del gasto calórico.
* **Intensidad ↔ pasos:** un aumento de intensidad se traduce en más volumen de pasos.
* **Calorías ↔ pasos:** el volumen de pasos predice bien el gasto calórico.

> 📌 Estos valores corresponden al **código crudo de intensidad por minuto**. No deben
> confundirse con la correlación del `combined_activity_score` diario, que se construye
> en la sección 6.3 y se evalúa en la 6.7.

---
## 5. Cohortes y datos diarios

### 5.1. Ventana temporal

In [ ]:
daily_all = loading.load_activity_files()

first_day = daily_all["ActivityDay"].min()
last_day = daily_all["ActivityDay"].max()
period_days = (last_day - first_day).days + 1

print(f"\nPrimera instancia : {first_day.date()}")
print(f"Última instancia  : {last_day.date()}")
print(f"Días del periodo  : {period_days}")
print(f"Usuarios totales  : {daily_all['Id'].nunique()}")

### 5.2. Separación en cohortes por constancia de uso

Se separan tres grupos según los días con registro:

| Cohorte | Criterio | Uso en el análisis |
| :--- | :--- | :--- |
| **Uso activo** | ≥ 57 días (8 semanas + 1) | Análisis de comportamiento habitual (secciones 6 a 8) |
| **Uso poco recurrente** | 36 – 56 días | Oportunidad de *re-engagement* (sección 9) |
| **Muy poco uso** | ≤ 35 días | Excluido: no hay suficiente historial para inferir hábitos |

> ⚠️ **Este filtro introduce un sesgo de selección que hay que declarar.** La cohorte
> principal se define por tener **pocos días faltantes**, de modo que cualquier medida
> de completitud sobre ella será alta **por construcción**. El dato relevante para el
> negocio no es lo completa que esté la cohorte, sino **cuánta gente hubo que excluir**.

In [ ]:
cohorts = features.cohort_summary(daily_all, period_days)
active_raw, reengagement_raw, dropped_raw = features.split_cohorts(daily_all)

ax = plots.cohort_flow_chart(cohorts)
plots.caption(
    ax,
    f"{int(cohorts.loc['Uso poco recurrente', 'usuarios'] + cohorts.loc['Muy poco uso', 'usuarios'])} "
    f"de {int(cohorts['usuarios'].sum())} usuarios quedaron fuera de la cohorte principal "
    "por uso intermitente del dispositivo.",
)
plots.save(ax, "cohortes.png")
plt.show()

cohorts

---
## 6. Análisis de la cohorte de uso activo

### 6.1. Ingeniería de variables

La variable central del análisis es el **puntaje de actividad combinada**:

$$\text{combined\_activity\_score} = 2 \times \text{VeryActiveMinutes} + \text{ModeratelyActiveMinutes}$$

El coeficiente 2 **no es arbitrario**: es exactamente la equivalencia que define la OMS
(75 min vigorosos ≡ 150 min moderados). Resuelve un problema real: dos personas con
40 minutos moderados y 20 vigorosos respectivamente realizan el mismo esfuerzo efectivo,
pero cualquier ranking por minutos totales pondría a la primera por delante.

También se descomponen las calorías en **basales** y **de actividad**, usando la mediana
de calorías por minuto en reposo **de cada persona** (no un valor poblacional), lo que
personaliza la estimación.

In [ ]:
active = features.add_processing_columns(active_raw, df_minutes)
reengagement = features.add_processing_columns(reengagement_raw, df_minutes)

nuevas = [c for c in active.columns if c not in daily_all.columns]
print("Columnas derivadas:", nuevas)

# Comprobación de integridad: cada día cae en exactamente una banda de intensidad
assert active[cfg.INTENSITY_FLAGS].sum(axis=1).eq(1).all(), "Banda de intensidad ambigua"
print("\n✓ Cada día está asignado a exactamente una banda de cumplimiento")

active[["ActivityDay", "Steps", "VeryActiveMinutes", "ModeratelyActiveMinutes",
        "combined_activity_score", "Calories", "calories_sedentary",
        "calories_activity"]].head()

### 6.2. Agregación por persona

Se resume cada persona en una fila. Se usa la **mediana** —no la media— para los pasos y
el puntaje de actividad: es robusta a días atípicos. Alguien que corre una maratón un
domingo no debe reclasificarse de «sedentario» a «muy activo» por un solo día, porque la
campaña de marketing le llegaría equivocada.

In [ ]:
user_activity = active.groupby("Id").agg(
    initial_date=("ActivityDay", "min"),
    latest_date=("ActivityDay", "max"),
    count_records=("ActivityDay", "count"),
    missing_days=("ActivityDay", lambda x, total=period_days: total - x.count()),

    full_days=("minutes_instances", lambda x: (x == cfg.MINUTES_PER_DAY).sum()),
    incomplete_days=("minutes_instances", lambda x: (x != cfg.MINUTES_PER_DAY).sum()),

    # Calorías: totales y la parte atribuible a actividad
    sum_all_calories=("Calories", "sum"),
    median_calories_activity=("calories_activity", "median"),

    median_steps=("Steps", "median"),
    median_activity_score=("combined_activity_score", "median"),
    total_activity_score=("combined_activity_score", "sum"),

    # Días en cada banda de cumplimiento de intensidad
    **{flag: (flag, "sum") for flag in cfg.INTENSITY_FLAGS},

    # Sueño
    days_of_sleep=("has_sleep_record", "sum"),
    healthy_sleep_nights=("healthy_sleep", "sum"),
)

user_activity

### 6.3. Completitud de datos — y por qué esta cifra engaña

La cohorte activa tiene una completitud muy alta. **Ese resultado estaba garantizado por
el filtro**: se seleccionaron las personas con más días registrados y después se midió
cuántos días les faltaban. Es un razonamiento circular, y presentarlo como un hallazgo
positivo sería un error de lectura.

El dato con valor de negocio es el complementario:

In [ ]:
expected = len(user_activity) * period_days
registered = int(user_activity["count_records"].sum())
excluded = int(cohorts.loc["Uso poco recurrente", "usuarios"]
               + cohorts.loc["Muy poco uso", "usuarios"])
total_users = int(cohorts["usuarios"].sum())

print("COHORTE ACTIVA (alta por construcción, no es un hallazgo)")
print(f"  {registered:,} de {expected:,} días-usuario con registro = "
      f"{100 * registered / expected:.1f} %\n")

print("DATASET COMPLETO (este sí es el hallazgo de retención)")
print(f"  {excluded} de {total_users} usuarios ({100 * excluded / total_users:.0f} %) "
      f"usaron el dispositivo de forma tan intermitente")
print("  que no permiten analizar comportamiento habitual.")
print(f"  De ellos, {int(cohorts.loc['Uso poco recurrente', 'usuarios'])} conservan "
      "datos suficientes para una campaña de re-enganche (sección 9).")

### 6.4. Registros de sueño: dato observado frente a causa

Aproximadamente la mitad de los días carece de registro de sueño. La lectura tentadora es
«el sensor falla», pero **hay al menos dos explicaciones incompatibles** y el dato bruto
no distingue entre ellas:

| Hipótesis | ¿Comprobable con estos datos? |
| :--- | :--- |
| El sensor falla o el modelo no tiene la función | Parcialmente |
| **La persona no llevaba el dispositivo de noche** | ✅ Sí — mirando si hay registros por minuto en horario nocturno |

La celda siguiente **contrasta la hipótesis** en lugar de asumirla: para cada noche sin
registro de sueño, comprueba si el dispositivo estaba registrando actividad entre la
01:00 y las 05:00.

In [ ]:
n_sleep = int(user_activity["days_of_sleep"].sum())
n_healthy = int(user_activity["healthy_sleep_nights"].sum())

print(f"Días con registro de sueño : {n_sleep:,} de {registered:,} "
      f"= {100 * n_sleep / registered:.2f} %")
print(f"De ellos, con >= 7 h       : {n_healthy:,} = {100 * n_healthy / max(n_sleep, 1):.1f} %\n")

# --- Contraste de la hipótesis: ¿estaba puesto el dispositivo esa noche? ---
night = df_minutes[df_minutes["ActivityMinute"].dt.hour.isin([1, 2, 3, 4])]
night_coverage = (
    night.assign(day=night["ActivityMinute"].dt.normalize())
    .groupby(["Id", "day"])
    .size()
    .rename("night_minutes")
    .reset_index()
)

check = (
    active[["Id", "ActivityDay", "has_sleep_record"]]
    .rename(columns={"ActivityDay": "day"})
    .merge(night_coverage, on=["Id", "day"], how="left")
    .fillna({"night_minutes": 0})
)

no_sleep = check[~check["has_sleep_record"]]
worn = int((no_sleep["night_minutes"] > 0).sum())

print(f"Noches SIN registro de sueño: {len(no_sleep):,}")
print(f"  · con actividad nocturna registrada (dispositivo puesto) : {worn:,} "
      f"({100 * worn / max(len(no_sleep), 1):.1f} %)  -> fallo de detección")
print(f"  · sin actividad nocturna (no lo llevaba puesto)          : "
      f"{len(no_sleep) - worn:,}  -> oportunidad de uso, no fallo")

### 6.5. Validez del puntaje de actividad combinada

Aquí se evalúa la **métrica derivada** (no el código crudo de intensidad de la sección 4.2).
La pregunta es si captura esfuerzo real sin ser redundante con los pasos: si correlacionara
casi perfectamente con `Steps`, no aportaría información nueva.

> ⚠️ **Advertencia metodológica.** Estas correlaciones se calculan sobre **días-usuario**.
> No son observaciones independientes: son ≈ 61 medidas repetidas de cada una de 26
> personas. Eso es **pseudorreplicación** e infla la aparente solidez del resultado.
> La segunda tabla recalcula la correlación agregando por persona, donde el n sí es el
> número de sujetos. Ambas se reportan como **descriptivas**, no inferenciales.

In [ ]:
corr_cols = ["Calories", "Steps", "SedentaryMinutes", "LightActiveMinutes",
             "ModeratelyActiveMinutes", "VeryActiveMinutes", "combined_activity_score"]

print("Correlaciones diarias (día-usuario)")
display(active[corr_cols].corr().round(3))

print("\nCorrección por medidas repetidas — puntaje combinado vs. calorías")
display(stats.correlation_by_unit(active, ["combined_activity_score", "Calories"]))

print("Corrección por medidas repetidas — puntaje combinado vs. pasos")
display(stats.correlation_by_unit(active, ["combined_activity_score", "Steps"]))

### 6.6. Volumen frente a intensidad

El diagrama de dispersión muestra la relación entre volumen de pasos y puntaje de
intensidad efectiva. Cada punto es un día registrado.

* **Tendencia ascendente:** los pasos son el principal impulsor de la intensidad efectiva.
* **Densidad cerca del origen:** confirma la existencia de días de muy baja actividad.
* **Mayor dispersión en los rangos altos:** por encima de ~15 000 pasos la eficiencia
  varía mucho entre personas. Algunas logran alta intensidad con menos pasos; otras
  necesitan mucho más volumen para el mismo efecto. **Esto es lo que justifica segmentar
  por calidad (intensidad) y no solo por cantidad (pasos).**

In [ ]:
fig, ax = plt.subplots()
sns.scatterplot(x="Steps", y="combined_activity_score", data=active,
                alpha=0.45, s=22, color=cfg.SEGMENT_COLORS["Saludable"],
                edgecolor="none", ax=ax)
sns.regplot(x="Steps", y="combined_activity_score", data=active, scatter=False,
            color=cfg.SEGMENT_COLORS["Sedentario"], line_kws={"lw": 2.5}, ax=ax)

ax.set_title("Pasos diarios frente a puntaje de intensidad efectiva")
ax.set_xlabel("Total de pasos diarios")
ax.set_ylabel("Puntaje de actividad combinada (minutos efectivos)")

r = active["Steps"].corr(active["combined_activity_score"])
plots.caption(ax, f"Correlación de Pearson r = {r:.2f} sobre {len(active):,} días-usuario "
                  f"({active['Id'].nunique()} personas; ver advertencia en 6.5).")
plt.show()

---
## 7. Segmentación de clientes

### 7.1. Cruce de volumen e intensidad

Se clasifica a cada persona en dos dimensiones a la vez: su **mediana de pasos** y su
**mediana de puntaje de intensidad**. Cruzarlas —en lugar de usar una sola métrica— es
la decisión de diseño central del análisis, y es lo que permite detectar perfiles que una
segmentación unidimensional clasificaría mal.

In [ ]:
user_activity = segmentation.categorize_users(user_activity)
matrix = segmentation.segmentation_matrix(user_activity)

ax = plots.segmentation_heatmap(matrix)
plots.caption(ax, "Distribución de la cohorte activa por comportamiento típico.")
plots.save(ax, "segmentacion_heatmap.png")
plt.show()

matrix

### 7.2. Los cuatro segmentos de negocio

Agrupando los cuadrantes en dos niveles por eje se obtienen cuatro perfiles accionables:

* **Pasos:** mínimo (`Sedentario`, `Mínimo`, `Leve`) frente a activo (`Óptimo`, `Muy activo`)
* **Intensidad:** mínima (`Sedentario`, `Mínimo saludable`) frente a óptima (`Objetivo óptimo`, `Objetivo máximo`)

| Segmento | Perfil | Interpretación de negocio |
| :--- | :--- | :--- |
| **Sedentario** | Pocos pasos, baja intensidad | Puede querer monitorizar signos vitales o mejorar hábitos. Usa el dispositivo casi todo el tiempo, así que la monitorización continua es lo valioso |
| **Viajero** | Muchos pasos, baja intensidad | Mucho desplazamiento sin exigencia corporal |
| **Fuerte** | Pocos pasos, alta intensidad | **El hallazgo del cruce bidimensional.** Probable entrenamiento de fuerza, ciclismo o natación: actividades que un podómetro subestima. Una segmentación solo por pasos los clasificaría como «población en riesgo» y les enviaría el mensaje equivocado |
| **Saludable** | Muchos pasos, alta intensidad | Alto compromiso y rendimiento sostenido |

> 📌 Los porcentajes se acompañan **siempre** del conteo y de un intervalo de confianza
> del 95 % (Wilson). Con n = 26, una sola persona vale ≈ 3,8 puntos porcentuales: reportar
> «57,7 %» sugeriría una precisión que la muestra no puede sostener.

In [ ]:
segments = segmentation.assign_segments(user_activity)
sizes = stats.segment_sizes_with_ci(segments)
sizes

### 7.3. Riesgo frente a rendimiento (mediana de días por persona)

La tabla siguiente muestra, para cada categoría de pasos, la **mediana de días** que sus
integrantes pasan en cada banda de cumplimiento de intensidad.

In [ ]:
# Reindexado a todas las categorías: groupby(observed=True) descarta las vacías
median_days = segmentation.flags_by_steps_category(user_activity, "median")
median_days

### 7.4. Cuantificación del impacto: volumen total de días

Frente a la mediana (comportamiento típico), la **suma total de días** cuantifica dónde se
concentra el volumen de inactividad y de esfuerzo máximo en la muestra.

In [ ]:
volume = segmentation.flags_by_steps_category(user_activity, "sum")

ax = plots.intensity_volume_bars(volume)
plots.caption(ax, "La tabla impresa arriba es la vista accesible equivalente de este gráfico.")
plots.save(ax, "volumen_dias.png")
plt.show()

volume

#### 7.4.1. Concentración del riesgo

El mayor volumen de inactividad se concentra en los segmentos de pasos bajos.

> Este hallazgo cuantifica el problema principal para Bellabeat: la inactividad se debe
> principalmente al **bajo volumen de pasos**, no a la falta de intensidad puntual.

#### 7.4.2. Concentración del rendimiento

La actividad de alta intensidad la produce casi en su totalidad el grupo que ya tiene alto
volumen de pasos, lo que es coherente con la correlación observada en la sección 6.5.

#### 7.4.3. Fiabilidad del segmento activo

Las personas de pasos altos acumulan muy pocos días de riesgo en comparación con las de
pasos bajos. Su volumen de inactividad es marginal, lo que refuerza la estabilidad del
segmento **Saludable**.

#### 7.4.4. El cliente Fuerte: menor movimiento, mayor intensidad

Existe un grupo minoritario con **intensidad alta pese a un volumen de pasos bajo**. Sugiere
actividad de bajo impacto en desplazamiento —fuerza, ciclismo, natación— donde la métrica
de pasos **subestima su esfuerzo real**.

> ⚠️ Con n = 3, este segmento es una **hipótesis a validar**, no una conclusión. Se reporta
> porque su implicación de negocio (no enviarles mensajes de «población en riesgo») es
> asimétrica: el coste de equivocarse tratándolos como sedentarios es alto.

---
## 8. Consistencia semanal por segmento

### 8.1. Construcción de la vista semanal

Se descartan las semanas incompletas. Las semanas completas se **derivan de los datos**
(aquellas con los 7 días presentes en la ventana), no se codifican a mano: así el análisis
sigue funcionando con cualquier otro rango de fechas.

El criterio de cumplimiento se declara explícitamente, porque hay dos y dan lecturas muy
distintas:

| Criterio | Umbral | Origen |
| :--- | :--- | :--- |
| **Objetivo óptimo** | ≥ 301 min/semana | Techo del rango OMS (300 min) |
| **Mínimo saludable** | ≥ 154 min/semana | Piso del rango OMS (150 min) |
| **Pasos** | ≥ 49 882 pasos/semana | Corte «Óptimo» del JACC × 7 |

Se reportan **ambos** niveles de intensidad. Medir solo contra el techo hace que la brecha
entre segmentos parezca más dramática de lo que es.

In [ ]:
days_per_week = active.groupby("Week_ISO")["ActivityDay"].nunique()
complete_weeks = days_per_week[days_per_week == cfg.DAYS_PER_WEEK].index
print(f"Semanas ISO completas derivadas de los datos: {sorted(complete_weeks)}")

weekly = (
    active[active["Week_ISO"].isin(complete_weeks)]
    .groupby(["Id", "Week_ISO"])
    .agg(
        count_records=("ActivityDay", "count"),
        sum_all_calories=("Calories", "sum"),
        mean_steps=("Steps", "mean"),
        sum_steps=("Steps", "sum"),
        combined_activity_score=("combined_activity_score", "sum"),
    )
    .reset_index()
)
# Solo semanas con los 7 días registrados para esa persona
weekly = weekly[weekly["count_records"] == cfg.DAYS_PER_WEEK]

weekly["meets_intensity_target"] = weekly["combined_activity_score"] >= cfg.WEEKLY_INTENSITY_TARGET
weekly["meets_intensity_minimum"] = weekly["combined_activity_score"] >= cfg.WEEKLY_INTENSITY_MINIMUM
weekly["meets_steps_target"] = weekly["sum_steps"] >= cfg.WEEKLY_STEPS_TARGET

print(f"Semanas-persona analizadas: {len(weekly)} · personas: {weekly['Id'].nunique()}")

### 8.2. Cumplimiento por segmento

Porcentaje medio de semanas en las que cada segmento alcanza cada objetivo.

In [ ]:
compliance = weekly.groupby("Id").agg(
    weeks=("Week_ISO", "count"),
    pct_intensity_target=("meets_intensity_target", "mean"),
    pct_intensity_minimum=("meets_intensity_minimum", "mean"),
    pct_steps_target=("meets_steps_target", "mean"),
)
for col in [c for c in compliance.columns if c.startswith("pct")]:
    compliance[col] = (compliance[col] * 100).round(1)

rows = []
for name, ids in segments.items():
    part = compliance[compliance.index.isin(ids)]
    rows.append({
        "segmento": name,
        "personas": len(ids),
        "intensidad_optimo_%": round(part["pct_intensity_target"].mean(), 1) if len(part) else None,
        "intensidad_minimo_%": round(part["pct_intensity_minimum"].mean(), 1) if len(part) else None,
        "pasos_%": round(part["pct_steps_target"].mean(), 1) if len(part) else None,
        "concluyente": "sí" if len(ids) >= 5 else "no (n < 5)",
    })

pd.DataFrame(rows).set_index("segmento")

### 8.3. Consistencia semanal

Cada panel muestra la **mediana** del segmento y su **rango intercuartílico**. La anchura de
la banda es la información relevante: ancha significa comportamiento errático, estrecha
significa consistente.

El eje Y está **compartido por fila**, de modo que los segmentos son directamente
comparables entre sí — algo que gráficos separados con escalas propias no permiten.

In [ ]:
fig = plots.weekly_small_multiples(weekly, segments)
plots.save(fig, "consistencia_semanal.png")
plt.show()

---
## 9. Personas con uso poco recurrente (oportunidad de *re-engagement*)

Registraron entre 36 y 56 días. Conservan al menos la mitad del periodo y sus días
registrados son de alta calidad, con muy pocos minutos faltantes: **su patrón problemático
es el uso intermitente del dispositivo, no la calidad del dato cuando lo usan**.

> ⚠️ La muestra es demasiado pequeña para identificar un patrón de actividad física común.
> Lo que sí se puede afirmar es el patrón de **uso**.

In [ ]:
reengagement_users = reengagement.groupby("Id").agg(
    count_records=("ActivityDay", "count"),
    missing_days=("ActivityDay", lambda x, total=period_days: total - x.count()),
    median_steps=("Steps", "median"),
    median_activity_score=("combined_activity_score", "median"),
    available_minutes=("minutes_instances", "sum"),
)

reengagement_users["missing_minutes"] = (
    reengagement_users["count_records"] * cfg.MINUTES_PER_DAY
    - reengagement_users["available_minutes"]
)
reengagement_users["pct_minutes_registered"] = (
    100 * reengagement_users["available_minutes"]
    / (reengagement_users["available_minutes"] + reengagement_users["missing_minutes"])
).round(2)
reengagement_users["pct_days_registered"] = (
    100 * reengagement_users["count_records"] / period_days
).round(2)

reengagement_users = segmentation.categorize_users(reengagement_users)

print(f"Días registrados observados : {reengagement_users['count_records'].min()}"
      f"-{reengagement_users['count_records'].max()}")
print(f"Minutos faltantes por día   : "
      f"{100 - reengagement_users['pct_minutes_registered'].max():.2f} %"
      f" - {100 - reengagement_users['pct_minutes_registered'].min():.2f} %")

reengagement_users

La segmentación se aplica también a esta cohorte. **Es el caso que rompía la
implementación anterior**: aquí varias categorías quedan vacías, y una selección por
posición (`iloc[:3, :2]`) habría asignado personas al segmento equivocado en silencio.
La selección por etiqueta lo maneja correctamente, y el `assert` interno garantiza que
nadie se pierda ni se duplique.

In [ ]:
display(segmentation.segmentation_matrix(reengagement_users))
{k: len(v) for k, v in segmentation.assign_segments(reengagement_users).items()}

---
## 10. Resumen y conclusiones

Las categorizaciones se basan en investigación en salud publicada (OMS y JACC), no en
umbrales populares.

### 10.1. Comportamiento de uso

La muestra se divide en tres cohortes de constancia (ver 5.2 y 6.3). En la cohorte de uso
activo predominan dos perfiles opuestos:

* **Sedentario:** conducta muy irregular, con actividad leve o nula; hay semanas con
  resultados saludables y otras con apenas movimiento. Pueden ser personas de oficina,
  estudiantes, trabajo en casa, o quienes empiezan a mejorar hábitos o necesitan control
  por motivos de salud. **Usan el dispositivo casi todo el tiempo**, así que la
  monitorización continua es lo que valoran.

* **Saludable:** gran actividad en movimiento e intensidad, con cumplimiento positivo en
  casi todas las semanas. Posiblemente personas deportistas, de gimnasio, o con trabajos
  que exigen movimiento constante.

### 10.2. Registro de sueño

Aproximadamente la mitad de los días carece de registro de sueño. La sección 6.4 contrasta
las dos explicaciones posibles —fallo de detección frente a no llevar el dispositivo de
noche—. **La distinción importa**, porque cada una lleva a una recomendación distinta:
mejorar el sensor, o dar razones para llevarlo puesto por la noche.

### 10.3. Respuesta a las preguntas guía

| Pregunta (sección 1.2) | Respuesta |
| :--- | :--- |
| ¿Se puede segmentar por consistencia? | Sí. El cruce volumen × intensidad produce cuatro segmentos, tres de ellos poblados |
| ¿Hay patrones de alta actividad con pocos pasos? | Sí: el segmento **Fuerte**. Es el hallazgo del enfoque bidimensional, aunque con n = 3 sigue siendo una hipótesis |
| ¿Quiénes son más y menos consistentes? | Los **Saludables** son consistentes semana a semana; los **Sedentarios** son erráticos (ver la anchura de las bandas en 8.3) |
| ¿Qué proporción aporta datos intermitentes? | Alrededor de una cuarta parte de la muestra tuvo que excluirse de la cohorte principal (ver 6.3) |

---
## 11. Recomendaciones

> ⚠️ **Separación explícita entre datos y supuestos.** El análisis determina **a quién**
> dirigirse y **qué comportamiento** tiene cada segmento. Los **atributos del producto**
> que se mencionan a continuación (materiales, autonomía, diseño) **no proceden de este
> análisis**: son el posicionamiento declarado de la marca y deben confirmarse con el
> equipo de producto y con la ficha técnica oficial antes de usarse en una campaña real.
> Se marcan como supuesto de negocio, no como hallazgo.

El análisis concluye que la muestra se concentra en los polos sedentario y muy activo.
La campaña debería destacar la comodidad de llevar el producto en todo momento:

1. **Resistencia de materiales** — *supuesto de producto por confirmar.*
   Relevante para los segmentos **Fuerte** y **Saludable**, que exponen el dispositivo a
   roces y golpes en trabajos físicos o gimnasio.

2. **Diseño discreto (comodidad social)** — *supuesto de producto por confirmar.*
   Relevante para el segmento **Sedentario**, con posibles trabajos de oficina, donde el
   dispositivo debe integrarse en el entorno laboral sin sacrificar estilo.

3. **Autonomía de batería** — *supuesto de producto por confirmar.*
   Reduce la fricción de recarga. Cuanto menos haya que quitarse el dispositivo, más
   completo es el registro — algo que los datos **sí** respaldan: la completitud del
   registro es la variable que separa a las tres cohortes de la sección 5.2.

4. **Comodidad para dormir (énfasis en sueño)** — *parcialmente respaldado por los datos.*
   La sección 6.4 cuantifica cuántas noches carecen de registro de sueño **y distingue si
   el dispositivo estaba puesto**. Si predomina el no-uso nocturno, la palanca es dar
   razones para llevarlo de noche; si predomina el fallo de detección, la palanca es el
   sensor. **Conviene ejecutar esa comprobación antes de decidir el mensaje.**

5. **Campaña de re-enganche** — *respaldado por los datos.*
   Dirigida a la cohorte de uso poco recurrente (sección 9), que conserva datos de calidad
   pero uso intermitente. Mensaje propuesto: enfocado en el **monitoreo holístico**
   (sueño, ciclos, bienestar) y no solo en pasos o intensidad, para dar valor a los días
   de descanso.

### 11.1. Cómo se mediría el éxito

Un test A/B por segmento, porque el punto del análisis es que **el mensaje debe diferir
según el segmento**:

* **KPI primario:** días activos por persona a los 30 días, contra un grupo de control del
  mismo segmento.
* **KPI secundario:** tasa de uso nocturno (valida directamente la recomendación 4).
* **Métrica de contención:** tasa de bajas o desinstalación, para detectar si los mensajes
  de reactivación resultan intrusivos.

El criterio de éxito debe fijarse **antes** de lanzar (por ejemplo, +8 puntos de días
activos en el segmento Sedentario con potencia del 80 %), lo que determina el tamaño de
muestra necesario por rama.

---
## 12. Anexo: trazabilidad

### 12.1. Cifras citadas en el README

Las cifras del README se generan desde esta celda **y se comparan automáticamente** con lo
que el README afirma. Si alguna diverge, la celda lo avisa: así una cifra publicada no puede
quedar desincronizada del análisis sin que nadie lo note.

> ⚠️ **Nota sobre la correlación por minuto (0,91).** El filtro de fechas anterior comparaba
> cadenas de texto, lo que **excluía todos los minutos del último día de cada rango**. Al
> corregirlo entran más minutos en el cálculo, así que este valor puede moverse ligeramente
> respecto al publicado. Las correlaciones **diarias** (0,76 y 0,85) no se ven afectadas.
> Revisa el aviso que imprime la celda y actualiza el README si hace falta.

In [ ]:
n = len(user_activity)
r_cal = active["combined_activity_score"].corr(active["Calories"])
r_steps = active["combined_activity_score"].corr(active["Steps"])
risk_days = int(volume.loc[cfg.STEPS_LOW, cfg.INTENSITY_LABELS[0]].sum())
elite_days = int(volume.loc[cfg.STEPS_HIGH, cfg.INTENSITY_LABELS[3]].sum())

print("=" * 66)
print("CIFRAS PARA EL README — copiar tal cual")
print("=" * 66)
print(f"Cohorte analizada                          : n = {n} (de {total_users})")
print(f"r  puntaje combinado <-> calorías (diario) : {r_cal:.2f}")
print(f"r  puntaje combinado <-> pasos    (diario) : {r_steps:.2f}")
print(f"r  intensidad <-> calorías (por minuto)    : "
      f"{corr_minute.loc['Intensity', 'Calories']:.2f}")
print("-" * 66)
for name, ids in segments.items():
    print(f"{name:<12}: {stats.proportion_report(len(ids), n)}")
print("-" * 66)
print(f"Días bajo el umbral mínimo (pasos bajos)   : {risk_days}")
print(f"Días de máximo rendimiento (pasos altos)   : {elite_days}")
print(f"Días con registro de sueño                 : "
      f"{100 * n_sleep / registered:.2f} %")
print("=" * 66)

# --- Guardián: comprueba que el README sigue diciendo la verdad ---
PUBLICADO = {
    "r puntaje combinado <-> calorías": (round(r_cal, 2), 0.76),
    "r puntaje combinado <-> pasos": (round(r_steps, 2), 0.85),
    "r intensidad <-> calorías (minuto)": (
        round(corr_minute.loc["Intensity", "Calories"], 2), 0.91),
    "días de riesgo (pasos bajos)": (risk_days, 325),
    "días de élite (pasos altos)": (elite_days, 806),
    "tamaño de la cohorte": (n, 26),
}
divergencias = [(k, calc, pub) for k, (calc, pub) in PUBLICADO.items() if calc != pub]

if divergencias:
    print("\n⚠️  DIVERGENCIA CON EL README — actualízalo antes de publicar:")
    for k, calc, pub in divergencias:
        print(f"    {k}: calculado {calc} · README dice {pub}")
else:
    print("\n✓ Todas las cifras publicadas en el README coinciden con este análisis.")

### 12.2. Entorno de ejecución

Deja constancia de con qué versiones se produjeron estos resultados.

In [ ]:
import matplotlib
import seaborn
import bellabeat

print(f"Ejecutado    : {pd.Timestamp.now():%Y-%m-%d %H:%M}")
print(f"Python       : {sys.version.split()[0]}")
for mod in (pd, np, matplotlib, seaborn):
    print(f"{mod.__name__:<13}: {mod.__version__}")
print(f"bellabeat    : {bellabeat.__version__}")